# Create W. M. Keck Foundation Awards

**W. M. Keck Foundation** (funder_id `4320306159`, IAMHRF, priority `349`).
wmkeck.org WP-media REST → 45 annual abstract PDFs → 318 grants (Southern
California 162, Science & Engineering 78, Medical Research 78). 100%
institution/USD-amount/year; PI 49% (Medical-Research/Sci-Eng are PI-level,
Southern-California is org-level); titles only on 2024+ MR/SE PDFs (15%) — older
PDFs carry no title, so display_name falls back to `<program>: <institution>`.

Parquet: `s3://openalex-ingest/awards/keck/keck_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.keck_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/keck/keck_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.keck_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.keck_raw LIMIT 5;

## Step 2: Create Keck Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.keck_awards
USING delta
AS
WITH
keck_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306159  -- W. M. Keck Foundation
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title), ''), CONCAT(g.program, ': ', g.institution)) as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'USD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.program as funder_scheme,
        'keck' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.keck_raw g
    CROSS JOIN keck_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'keck' AND priority = 349;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    349 as priority
FROM openalex.awards.keck_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(display_name) has_title,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(amount) has_amount, COUNT(lead_investigator) has_pi,
  ROUND(SUM(amount)/1000000,1) total_musd
FROM openalex.awards.keck_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt, ROUND(SUM(amount)/1000000,1) musd
FROM openalex.awards.keck_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='keck' AND priority=349;